<p>Indic Conformer models (typically from the AI4Bharat NeMo or ESPnet ecosystems) are high-performance ASR architectures. Quantizing them and moving to ONNX can significantly reduce latency and disk footprint, making them suitable for real-time edge deployment.

Since most Indic Conformer models are based on the NeMo framework, the standard path is to export them to ONNX first and then apply Post-Training Quantization (PTQ) using ONNX Runtime.</p>

<p><b>1. Exporting Indic Conformer to ONNX</b>


If you are using the AI4Bharat NeMo version, you should use the built-in export functionality. This handles the complex Conformer sub-modules (Attention, Convolutions) correctly.</p>

In [4]:
import torch
import nemo.collections.asr as nemo_asr

# 1. Load your pretrained model
restore_path = "indic_conformer_model.nemo"
model = nemo_asr.models.EncDecCTCModel.restore_from(restore_path)

# 2. Export to ONNX
# This creates 'indic_conformer.onnx'
model.export("indic_conformer.onnx", check_trace=True)

ModuleNotFoundError: No module named 'nemo'

<p><b>2. Quantizing the ONNX Model</b>


We use onnxruntime.quantization to perform Dynamic Quantization. This is usually the best choice for ASR models like Conformer, as it strikes a balance between performance and accuracy without needing a huge calibration dataset.</p>

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Path to your FP32 ONNX model
model_fp32 = "indic_conformer.onnx"
# Path to save the INT8 model
model_quant = "indic_conformer_int8.onnx"

# Quantize weights to INT8
quantize_dynamic(
    model_input=model_fp32,
    model_output=model_quant,
    weight_type=QuantType.QInt8
)

print(f"Quantization complete. Saved to: {model_quant}")

<p><b>3. Inference with the Quantized ONNX Model</b>


Once exported and quantized, you can run inference using onnxruntime. Note that ASR models require a pre-processor (to turn audio into Mel-spectrograms) and a post-processor (to turn logits into text).</p>

In [ ]:
import onnxruntime as ort
import numpy as np

# Load the session
session = ort.InferenceSession("indic_conformer_int8.onnx", providers=['CPUExecutionProvider'])

# Prepare dummy input (audio signal length should match model expectations)
# Usually (Batch, Time) or (Batch, Feat, Time) depending on the export
input_name = session.get_inputs()[0].name
dummy_audio = np.random.randn(1, 16000).astype(np.float32) 

# Run inference
logits = session.run(None, {input_name: dummy_audio})
print("Logits shape:", logits[0].shape)

<p><b>Key Considerations for Indic Conformer</b>


Decoding Strategy: Indic Conformer supports both CTC and RNN-T decoding. If you are using RNN-T, the ONNX export is more complex because it involves multiple files (Encoder, Decoder, and Joint).

Audio Pre-processing: Don't forget that the ONNX model expects processed features (like log-Mel filterbanks) unless you exported the model with the pre-processor included.

Hardware Acceleration: If you have an NVIDIA GPU, use CUDAExecutionProvider in the InferenceSession for a 5x-10x speedup over the CPU.</p>

<p>Building an end-to-end inference pipeline for an Indic Conformer ASR model involves three major stages: Audio Preprocessing (converting sound to Mel-spectrograms), the ONNX Model execution, and Decoding (converting model outputs into Indian language text).

Here is the complete workflow to move from a NeMo .nemo checkpoint to a quantized, production-ready ONNX pipeline.</p>

<p><b>1. Export and Quantize the Model</b>


First, we export the NeMo model to ONNX format and apply dynamic quantization to reduce the size (typically from ~500MB to ~130MB for a base Conformer).</p>

In [ ]:
import nemo.collections.asr as nemo_asr
from onnxruntime.quantization import quantize_dynamic, QuantType

# 1. Export to ONNX
model = nemo_asr.models.EncDecCTCModel.restore_from("indic_conformer.nemo")
model.export("indic_conformer.onnx")

# 2. Dynamic Quantization (Targeting INT8)
quantize_dynamic(
    model_input="indic_conformer.onnx",
    model_output="indic_conformer_int8.onnx",
    weight_type=QuantType.QInt8
)

<p><b>2. The Inference Pipeline</b>


Unlike BERT, ASR models cannot take raw text; they require signal processing. We use librosa for audio handling and onnxruntime for the math.</p>

import numpy as np
import librosa
import onnxruntime as ort

class IndicASRPipeline:
    def __init__(self, model_path, vocab):
        # Load the quantized model
        self.session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.vocab = vocab # List of characters/tokens for the specific language
        
    def preprocess(self, audio_path):
        # Indic Conformer usually expects 16kHz mono audio
        audio, _ = librosa.load(audio_path, sr=16000)
        # Normalize audio
        audio = audio / np.max(np.abs(audio))
        # Add batch dimension
        return np.expand_dims(audio, axis=0).astype(np.float32)

    def decode(self, logits):
        # Greedy Decoding: Pick the most likely character at each time step
        best_path = np.argmax(logits, axis=-1)[0]
        
        # CTC logic: Remove consecutive duplicates and 'blank' tokens (usually index 0 or len(vocab))
        tokens = []
        prev_token = None
        blank_idx = len(self.vocab) # Common convention in NeMo
        
        for t in best_path:
            if t != prev_token and t < len(self.vocab):
                tokens.append(self.vocab[t])
            prev_token = t
            
        return "".join(tokens)

    def transcribe(self, audio_path):
        processed_audio = self.preprocess(audio_path)
        
        # Run ONNX inference
        input_name = self.session.get_inputs()[0].name
        logits = self.session.run(None, {input_name: processed_audio})[0]
        
        return self.decode(logits)

# Usage Example:
# indic_vocab = ["अ", "आ", "इ", ...] 
# pipeline = IndicASRPipeline("indic_conformer_int8.onnx", indic_vocab)
# print(pipeline.transcribe("sample_hindi.wav"))

<p><b>3. Lightweight Alternative: onnx-asr</b>


If you don't want to manually write the preprocessing and CTC logic, a new library in 2025/2026 called onnx-asr has become the standard for running Indic models on the edge. It bundles the vocabulary and Mel-spectrogram logic automatically.</p>

In [ ]:
import onnx_asr

# Load the model (Supports AI4Bharat and NVIDIA Indic models)
model = onnx_asr.load_model("indic-conformer-hi-ctc", quantization="int8")

# Single command transcription
text = model.recognize("audio.wav")
print(f"Transcription: {text}")

<p>Why Quantize Indic Conformer?
CPU Real-time: A quantized Conformer can often run 3x–5x faster than real-time on a standard laptop CPU.

Memory: It reduces RAM usage, allowing you to run ASR alongside other processes (like a translation model) on a 4GB or 8GB device.

Indic Nuance: Quantization rarely impacts Word Error Rate (WER) for high-resource Indian languages (Hindi, Tamil, Bengali) by more than 0.2–0.5%.</p>